In [1]:
import pandas as pd
import numpy as np
import librosa
import os
from tqdm import tqdm

df = pd.read_csv("data/labels.csv")

MAX_LEN = 130   # fixed number of time frames per clip (for LSTM input)
N_MFCC = 13

def extract_features(filepath):
    y, sr = librosa.load(filepath, sr=22050)
    y, _ = librosa.effects.trim(y)  # remove silence

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    pitch, _ = librosa.piptrack(y=y, sr=sr)
    pitch_mean = np.mean(pitch[pitch > 0]) if np.any(pitch > 0) else 0
    energy = librosa.feature.rms(y=y)

    # Pad or truncate MFCC to fixed length for LSTM input
    if mfcc.shape[1] < MAX_LEN:
        pad_width = MAX_LEN - mfcc.shape[1]
        mfcc_padded = np.pad(mfcc, ((0,0),(0,pad_width)), mode='constant')
    else:
        mfcc_padded = mfcc[:, :MAX_LEN]

    # Aggregated (statistical) features for baseline model
    agg_features = np.concatenate([
        np.mean(mfcc, axis=1), np.std(mfcc, axis=1),
        [pitch_mean],
        [np.mean(energy)], [np.std(energy)]
    ])

    return mfcc_padded.T, agg_features  # sequence (for LSTM), aggregated (for baseline)

sequence_data = []
aggregated_data = []
labels = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    try:
        seq, agg = extract_features(row["filepath"])
        sequence_data.append(seq)
        aggregated_data.append(agg)
        labels.append(row["label"])
    except Exception as e:
        print("Skipped:", row["filepath"], e)

X_seq = np.array(sequence_data)
X_agg = np.array(aggregated_data)
y = np.array(labels)

np.save("data/X_seq.npy", X_seq)
np.save("data/X_agg.npy", X_agg)
np.save("data/y.npy", y)

print("Sequence data shape:", X_seq.shape)
print("Aggregated data shape:", X_agg.shape)
print("Labels shape:", y.shape)
print("Features saved successfully!")

  0%|          | 0/672 [00:00<?, ?it/s]C:\Users\SATYA\OneDrive\Desktop\voice-stress-detection-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 672/672 [00:42<00:00, 15.91it/s]

Sequence data shape: (672, 130, 13)
Aggregated data shape: (672, 29)
Labels shape: (672,)
Features saved successfully!
